# Unified A/A Test Framework for Synthetic Control Validation

## Purpose

Validates three synthetic control methodologies used across DE, NL, and EN markets:
- **V3** Customer Lookalike (with promo orders)
- **V2** Customer Lookalike (with city_type) — stub, enable when ready
- **City Lookalike** (MAVERICK KNN + CausalImpact)

Run against historical non-campaign periods. Expected result: **zero uplift**.
Any non-zero result quantifies the methodology's inherent measurement bias.

## Pass/Fail Criteria

| Verdict | Condition |
|---------|----------|
| **PASS** | |mean uplift| ≤ 2% AND 95% CI contains zero |
| **WARNING** | |mean uplift| > 2% OR 95% CI excludes zero |
| **HARD FAIL** | |mean uplift| > 5% in any period |

Results are reported at **total** level and **per customer base segment**.

---
## 1. Imports & Authentication

In [ ]:
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import time
import warnings
import gc

warnings.filterwarnings('ignore', category=FutureWarning)
sns.set_style('whitegrid')

# --- Authentication (Colab or local) ---
try:
    from google.colab import auth
    auth.authenticate_user()
    print('Authenticated via Colab')
except ImportError:
    print('Not running in Colab — using local credentials')

from google.cloud import bigquery

PROJECT_ID = 'just-data-gci-dev'
client = bigquery.Client(project=PROJECT_ID)
print(f'Connected to project: {PROJECT_ID}')

# --- Optional: CausalImpact for City Lookalike ---
try:
    from causalimpact import CausalImpact
    CAUSAL_IMPACT_AVAILABLE = True
    print('CausalImpact available')
except ImportError:
    CAUSAL_IMPACT_AVAILABLE = False
    print('WARNING: CausalImpact not installed. City Lookalike A/A will be skipped.')
    print('Install with: pip install tfcausalimpact')

---
## 2. Configuration

In [ ]:
# ============================================================
# A/A TEST CONFIGURATION
# ============================================================

# --- Techniques to run ---
RUN_V3 = True
RUN_V2 = False   # Requires city_type sourcing — set True when ready
RUN_CITY = True  # Requires CausalImpact installed

# --- Markets (run ONE at a time on Colab free to avoid OOM) ---
COUNTRIES = ['DE']  # Change to ['NL'] for second run

# --- Customer-level A/A settings ---
N_SEEDS = 5         # Reduced from 10 — still enough for t-test (5 seeds x 2 windows = 10 runs)
TREATMENT_SHARE = 0.5

# --- Bias thresholds ---
BIAS_THRESHOLD_HARD = 0.05  # |uplift| > 5% = HARD FAIL
BIAS_THRESHOLD_WARN = 0.02  # |uplift| > 2% = WARNING

# --- Campaign name prefix ---
AA_CAMPAIGN_PREFIX = 'AA_UNIFIED'

# --- Per-market dummy windows ---
# Reduced to 2 windows per market to save memory on Colab free tier.
# Add more windows when running on Colab Pro or locally.
TIME_WINDOWS = {
    'DE': [
        {'start': '2024-06-01', 'end': '2024-06-30',
         'post_start': '2024-07-01', 'post_end': '2024-07-31'},
        {'start': '2024-08-01', 'end': '2024-08-31',
         'post_start': '2024-09-01', 'post_end': '2024-09-30'},
    ],
    'NL': [
        {'start': '2024-06-01', 'end': '2024-06-30',
         'post_start': '2024-07-01', 'post_end': '2024-07-31'},
        {'start': '2024-08-01', 'end': '2024-08-31',
         'post_start': '2024-09-01', 'post_end': '2024-09-30'},
    ],
}

# --- Matching conditions (from production scripts) ---
LOOK_ALIKE_CONDITIONS_V3 = [
    'country', 'optin', 'campaign_start_date',
    'L14D_orders', 'L30D_orders', 'L90D_orders', 'L365D_orders',
]
LOOK_ALIKE_CONDITIONS_V2 = [
    'country', 'city_type', 'optin', 'campaign_start_date',
    'L14D_orders', 'L30D_orders', 'L90D_orders', 'L365D_orders', 'L365D_AOV_cat',
]

# --- City Lookalike config ---
CITY_TREATMENT_CANDIDATES = {
    'DE': ['Dortmund', 'Dresden', 'Essen'],
    'NL': [],
}
CITY_KNN_NEIGHBORS = 25
CITY_CORRELATION_THRESHOLD = 0.8
CITY_TIERS = ['TIER 2', 'TIER 3', 'TIER 4', 'TIER 5', 'TIER 6']

# --- BQ tables ---
AA_AUDIENCE_TABLE = (
    'just-data-warehouse.customer_intelligence.customer_lookalike_aa_audience'
)

# --- Segments (lowercase, matching fact_segmentation_scv_key) ---
BASE_SEGMENTS = [
    'new', 'reactivated', 'early', 'engaged',
    'lapsing', 'dormant', 'prospect', 'churned', 'unknown',
]

# --- Memory management ---
# Max lookalike matches to keep per treatment customer during exact match.
# Prevents cartesian explosion that kills Colab free tier.
MAX_LOOKALIKES_PER_TREATMENT = 10

print('Configuration loaded.')
print(f'Markets: {COUNTRIES}')
print(f'Techniques: V3={RUN_V3}, V2={RUN_V2}, City={RUN_CITY}')
total_customer_runs = sum(
    len(TIME_WINDOWS.get(c, [])) for c in COUNTRIES
) * N_SEEDS * (int(RUN_V3) + int(RUN_V2))
total_city_runs = sum(
    len(CITY_TREATMENT_CANDIDATES.get(c, []))
    * len(TIME_WINDOWS.get(c, []))
    for c in COUNTRIES
) if RUN_CITY else 0
print(f'Customer-level runs: {total_customer_runs}')
print(f'City-level runs: {total_city_runs}')

---
## 3. Campaign Overlap Validation

Checks three sources for campaign contamination:
1. `customer_lookalike_evaluation_audience`
2. `customer_lookalike_evaluation_audience_citytype`
3. OMT campaigns via `fact_offer`

In [ ]:
def validate_no_campaign_overlap(client, countries, time_windows):
    """Comprehensive campaign overlap check against audience tables and OMT offers."""
    checks = []
    for country in countries:
        for w in time_windows.get(country, []):
            window_label = f"{w['start']} to {w['post_end']}"

            # Lookalike audience tables
            for table in [
                'just-data-warehouse.customer_intelligence'
                '.customer_lookalike_evaluation_audience',
                'just-data-warehouse.customer_intelligence'
                '.customer_lookalike_evaluation_audience_citytype',
            ]:
                query = f"""
                SELECT COUNT(*) AS cnt
                FROM `{table}`
                WHERE country = '{country}'
                  AND (
                    campaign_start_date BETWEEN '{w['start']}' AND '{w['post_end']}'
                    OR campaign_end_date BETWEEN '{w['start']}' AND '{w['post_end']}'
                  )
                """
                try:
                    cnt = client.query(query).to_dataframe()['cnt'].iloc[0]
                except Exception:
                    cnt = -1
                checks.append({
                    'country': country, 'window': window_label,
                    'source': table.split('.')[-1], 'overlapping_rows': cnt,
                })

            # OMT campaigns via fact_offer
            omt_query = f"""
            SELECT COUNT(DISTINCT fo.orderid) AS cnt
            FROM `just-data-warehouse.dwh.fact_order` AS fo
            INNER JOIN `just-data-warehouse.dwh.fact_offer` AS fof
                ON fo.orderid = fof.orderid
            INNER JOIN `just-data-warehouse.dwh.dim_restaurant` AS dr
                ON fo.restaurantid = dr.restaurantid
            WHERE dr.country = '{country}'
              AND DATE(fo.orderdatetime)
                  BETWEEN '{w['start']}' AND '{w['post_end']}'
            """
            try:
                cnt = client.query(omt_query).to_dataframe()['cnt'].iloc[0]
            except Exception:
                cnt = -1
            checks.append({
                'country': country, 'window': window_label,
                'source': 'fact_offer (OMT)', 'overlapping_rows': cnt,
            })

    df_checks = pd.DataFrame(checks)
    overlaps = df_checks[df_checks['overlapping_rows'] > 0]
    if len(overlaps) > 0:
        print('WARNING: Overlap detected:')
        print(overlaps.to_string(index=False))
        print()
        print('Note: OMT overlaps may be market-wide and acceptable.')
        print('Lookalike audience overlaps indicate direct campaign contamination.')
    else:
        print('PASS: No campaign overlap detected for any window.')
    return df_checks

df_overlap = validate_no_campaign_overlap(client, COUNTRIES, TIME_WINDOWS)
df_overlap

---
## 4. Shared Utilities

In [ ]:
def cleanup_aa_audience(client, table_id, campaign_name):
    """Remove A/A audience rows after each run to prevent accumulation."""
    query = f"DELETE FROM `{table_id}` WHERE campaign_name = '{campaign_name}'"
    client.query(query).result()


def safe_divide(a, b):
    """Division returning NaN when denominator is zero."""
    return np.where(b > 0, a / b, np.nan)


def statistical_validation_single(values, label=''):
    """One-sample t-test + bootstrap 95% CI for a set of uplift values."""
    values = np.array(values)
    values = values[~np.isnan(values)]
    if len(values) < 3:
        return {
            'label': label, 'mean': np.nan, 'std': np.nan, 'n': len(values),
            't_stat': np.nan, 'p_value': np.nan,
            'ci_lower': np.nan, 'ci_upper': np.nan,
            'ci_contains_zero': None,
        }
    t_stat, p_value = stats.ttest_1samp(values, 0)
    rng = np.random.RandomState(42)
    boot_means = [
        rng.choice(values, size=len(values), replace=True).mean()
        for _ in range(10_000)
    ]
    ci_lower, ci_upper = np.percentile(boot_means, [2.5, 97.5])
    return {
        'label': label,
        'mean': values.mean(), 'std': values.std(), 'n': len(values),
        't_stat': t_stat, 'p_value': p_value,
        'ci_lower': ci_lower, 'ci_upper': ci_upper,
        'ci_contains_zero': bool(ci_lower <= 0 <= ci_upper),
    }


def assess_verdict(abs_mean, ci_contains_zero):
    """Apply pass/fail criteria to a single metric."""
    if np.isnan(abs_mean):
        return 'INSUFFICIENT_DATA'
    if abs_mean > BIAS_THRESHOLD_HARD:
        return 'HARD_FAIL'
    if abs_mean > BIAS_THRESHOLD_WARN:
        return 'WARNING'
    if ci_contains_zero is False:
        return 'WARNING'
    return 'PASS'

print('Utilities loaded.')

---
## 5. Audience Creation (Customer-level)

Joins segments via the canonical `dim_unique_customer_history` → `fact_segmentation_scv_key` path. Stratified random split by `base_segment`.

In [ ]:
def create_customer_aa_audience(client, country, window, seed,
                                treatment_share=0.5):
    """
    Select eligible customers, join segments point-in-time,
    randomly split into treatment/lookalike stratified by segment.
    """
    start = window['start']

    query = f"""
    SELECT
        fo.customerid,
        dc.country,
        CASE WHEN dcu.optin = true THEN 1 ELSE 0 END AS optin,
        COALESCE(s.base_segment, 'unknown') AS base_segment
    FROM `just-data-warehouse.dwh.fact_order` AS fo
    INNER JOIN `just-data-warehouse.dwh.dim_customer` AS dcu
        ON fo.customerid = dcu.customerid
    INNER JOIN `just-data-warehouse.dwh.dim_country` AS dc
        ON fo.countryid = dc.countryid
    LEFT JOIN `just-data-warehouse.core_dwh.dim_unique_customer_history` AS ch
        ON fo.customerid = ch.customer_id
        AND ch.snapshot_date = DATE('{start}')
    LEFT JOIN `just-data-warehouse.dwh.fact_segmentation_scv_key` AS s
        ON ch.scv_key = s.scv_key
        AND s.snapshot_date = DATE_SUB(DATE('{start}'), INTERVAL 1 DAY)
    WHERE dc.country = '{country}'
      AND DATE(orderdatetime) >= DATE_SUB(DATE('{start}'), INTERVAL 365 DAY)
      AND DATE(orderdatetime) < DATE('{start}')
      AND DATE(orderdatetime) >= DATE('2021-01-01')
    GROUP BY 1, 2, 3, 4
    HAVING SUM(nroforders) > 0
    """

    df = client.query(query).to_dataframe()

    # Stratified random split by segment
    rng = np.random.RandomState(seed)
    df['treatment'] = 0
    for segment in df['base_segment'].unique():
        mask = df['base_segment'] == segment
        df.loc[mask, 'treatment'] = rng.binomial(
            1, treatment_share, size=mask.sum()
        )

    campaign_name = f"{AA_CAMPAIGN_PREFIX}_{country}_{start}_seed{seed}"
    df['campaign_name'] = campaign_name
    df['campaign_start_date'] = pd.to_datetime(window['start']).date()
    df['campaign_end_date'] = pd.to_datetime(window['end']).date()
    df['post_period_start_date'] = pd.to_datetime(window['post_start']).date()
    df['post_period_end_date'] = pd.to_datetime(window['post_end']).date()

    n_t = (df['treatment'] == 1).sum()
    n_c = (df['treatment'] == 0).sum()
    print(f'  {country} | {start} | seed {seed}: '
          f'{len(df)} customers ({n_t} treat, {n_c} ctrl)')
    return df, campaign_name


def upload_aa_audience(client, df, table_id):
    """Write A/A audience to BigQuery.
    Note: base_segment is NOT uploaded — the existing BQ table
    does not have this column. Segments are joined during matching
    via dim_unique_customer_history -> fact_segmentation_scv_key.
    """
    upload_cols = [
        'customerid', 'country', 'campaign_name', 'treatment',
        'campaign_start_date', 'campaign_end_date',
        'post_period_start_date', 'post_period_end_date',
    ]
    df_upload = df[upload_cols].copy()
    schema = [
        bigquery.SchemaField('customerid', 'STRING'),
        bigquery.SchemaField('country', 'STRING'),
        bigquery.SchemaField('campaign_name', 'STRING'),
        bigquery.SchemaField('treatment', 'INTEGER'),
        bigquery.SchemaField('campaign_start_date', 'DATE'),
        bigquery.SchemaField('campaign_end_date', 'DATE'),
        bigquery.SchemaField('post_period_start_date', 'DATE'),
        bigquery.SchemaField('post_period_end_date', 'DATE'),
    ]
    job_config = bigquery.LoadJobConfig(
        schema=schema, write_disposition='WRITE_APPEND',
    )
    job = client.load_table_from_dataframe(
        df_upload, table_id, job_config=job_config
    )
    job.result()
    print(f'  Uploaded {len(df_upload)} rows to {table_id}')

print('Audience creation functions loaded.')

---
## 6. V3 Customer Lookalike Matching

Faithful adaptation from `customer_look_alike_evaluation_v3_with_promo_orders_bilal.py`.
Only changes: table reference, campaign_name parameter, added `base_segment` and `post_period_orders`.

In [ ]:
def run_matching_v3(client, country, date_str, campaign_name,
                    audience_table, look_alike_conditions):
    """V3 Customer Lookalike matching — BigQuery-native.

    Step 1+2 (features + exact match): BQ scripting with temp tables.
    Step 3 (KNN fallback): standalone BQ query + local sklearn.
    """

    # ---- Core feature subquery (reusable, no temp tables) ----
    feature_subquery = f"""
    SELECT *,
        NTILE(3) OVER (PARTITION BY country ORDER BY L365D_AOV) AS L365D_AOV_cat
    FROM (
        SELECT
            fo.customerid, lc.country,
            CASE WHEN dcu.optin = true THEN 1 ELSE 0 END AS optin,
            lc.campaign_start_date, lc.campaign_end_date,
            '{campaign_name}' AS campaign_name, lc.treatment,
            SUM(CASE WHEN DATE(orderdatetime) >= DATE_ADD(DATE(campaign_start_date), INTERVAL -7 DAY)
                      AND DATE(orderdatetime) < DATE(campaign_start_date)
                THEN nroforders ELSE 0 END) AS L7D_orders,
            SUM(CASE WHEN DATE(orderdatetime) >= DATE_ADD(DATE(campaign_start_date), INTERVAL -14 DAY)
                      AND DATE(orderdatetime) < DATE(campaign_start_date)
                THEN nroforders ELSE 0 END) AS L14D_orders,
            SUM(CASE WHEN DATE(orderdatetime) >= DATE_ADD(DATE(campaign_start_date), INTERVAL -30 DAY)
                      AND DATE(orderdatetime) < DATE(campaign_start_date)
                THEN nroforders ELSE 0 END) AS L30D_orders,
            SUM(CASE WHEN DATE(orderdatetime) >= DATE_ADD(DATE(campaign_start_date), INTERVAL -90 DAY)
                      AND DATE(orderdatetime) < DATE(campaign_start_date)
                THEN nroforders ELSE 0 END) AS L90D_orders,
            SUM(CASE WHEN DATE(orderdatetime) >= DATE_ADD(DATE(campaign_start_date), INTERVAL -30 DAY)
                      AND DATE(orderdatetime) < DATE(campaign_start_date)
                      AND restaurant_discount > 0
                THEN nroforders ELSE 0 END) AS L30D_promo_orders,
            SUM(CASE WHEN DATE(orderdatetime) >= DATE_ADD(DATE(campaign_start_date), INTERVAL -180 DAY)
                      AND DATE(orderdatetime) < DATE(campaign_start_date)
                THEN nroforders ELSE 0 END) AS L180D_orders,
            SUM(CASE WHEN DATE(orderdatetime) >= DATE_ADD(DATE(campaign_start_date), INTERVAL -365 DAY)
                      AND DATE(orderdatetime) < DATE(campaign_start_date)
                THEN nroforders ELSE 0 END) AS L365D_orders,
            SUM(CASE WHEN DATE(orderdatetime) >= DATE_ADD(DATE(campaign_start_date), INTERVAL -730 DAY)
                      AND DATE(orderdatetime) < DATE(campaign_start_date)
                THEN nroforders ELSE 0 END) AS L730D_orders,
            SAFE_DIVIDE(
                SUM(CASE WHEN DATE(orderdatetime) >= DATE_ADD(DATE(campaign_start_date), INTERVAL -365 DAY)
                          AND DATE(orderdatetime) < DATE(campaign_start_date)
                    THEN food_total ELSE 0 END),
                SUM(CASE WHEN DATE(orderdatetime) >= DATE_ADD(DATE(campaign_start_date), INTERVAL -365 DAY)
                          AND DATE(orderdatetime) < DATE(campaign_start_date)
                    THEN nroforders ELSE 0 END)
            ) AS L365D_AOV,
            SUM(CASE WHEN DATE(orderdatetime) > DATE(campaign_start_date)
                      AND DATE(orderdatetime) <= DATE(campaign_end_date)
                THEN nroforders ELSE 0 END) AS campaign_period_orders,
            SUM(CASE WHEN DATE(orderdatetime) > DATE(campaign_start_date)
                      AND DATE(orderdatetime) <= DATE(campaign_end_date)
                      AND restaurant_discount > 0
                THEN nroforders ELSE 0 END) AS campaign_period_promo_orders,
            SUM(CASE WHEN DATE(orderdatetime) > DATE(post_period_start_date)
                      AND DATE(orderdatetime) <= DATE(post_period_end_date)
                THEN nroforders ELSE 0 END) AS post_period_orders
        FROM `just-data-warehouse.dwh.fact_order` AS fo
        INNER JOIN `just-data-warehouse.dwh.dim_customer` AS dcu
            ON fo.customerid = dcu.customerid
        INNER JOIN `just-data-warehouse.dwh.dim_country` AS dc
            ON fo.countryid = dc.countryid
        INNER JOIN `{audience_table}` AS lc
            ON fo.customerid = lc.customerid
        WHERE DATE(orderdatetime) >= DATE_ADD(DATE(campaign_start_date), INTERVAL -730 DAY)
          AND DATE(orderdatetime) >= DATE('2020-01-01')
          AND lc.campaign_name = '{campaign_name}'
          AND lc.country = '{country}'
          AND DATE(lc.campaign_start_date) = '{date_str}'
        GROUP BY 1, 2, 3, 4, 5, 6, 7
    )
    """

    # Outlier filter clause (reused in multiple queries)
    outlier_filter = """
        L7D_orders <= 7 AND L14D_orders <= 14 AND L30D_orders <= 30
        AND L90D_orders <= 90 AND L180D_orders <= 180 AND L365D_orders <= 365
        AND L730D_orders <= 730 AND L30D_promo_orders = 0
    """

    # Build the match-key join condition
    match_keys = [c for c in look_alike_conditions
                  if c not in ('country', 'campaign_start_date')]
    match_join = ' AND '.join([f't.{c} = l.{c}' for c in match_keys])

    # ---- STEP 1+2: Features + exact match in BQ scripting ----
    exact_match_query = f"""
    CREATE TEMP TABLE features AS {feature_subquery};

    CREATE TEMP TABLE treatment AS
    SELECT * FROM features WHERE treatment = 1 AND {outlier_filter};

    CREATE TEMP TABLE lookalike AS
    SELECT * FROM features WHERE treatment = 0 AND {outlier_filter};

    CREATE TEMP TABLE exact_matches AS
    SELECT
        t.customerid AS customerid_treatment,
        t.country, t.optin, t.campaign_start_date,
        t.L30D_orders AS L30D_orders_treatment,
        t.campaign_period_orders AS campaign_period_orders_treatment,
        t.campaign_period_promo_orders AS campaign_period_promo_orders_treatment,
        t.post_period_orders AS post_period_orders_treatment,
        AVG(l.L30D_orders) AS L30D_orders_lookalike,
        AVG(l.campaign_period_orders) AS campaign_period_orders_lookalike,
        AVG(l.campaign_period_promo_orders) AS campaign_period_promo_orders_lookalike,
        AVG(l.post_period_orders) AS post_period_orders_lookalike,
        'exact_match' AS source
    FROM treatment t
    INNER JOIN lookalike l
        ON t.country = l.country
        AND t.campaign_start_date = l.campaign_start_date
        AND {match_join}
    GROUP BY 1, 2, 3, 4, 5, 6, 7, 8;

    SELECT *, 'exact' AS result_type FROM exact_matches
    UNION ALL
    SELECT
        t.customerid AS customerid_treatment,
        t.country, t.optin, t.campaign_start_date,
        t.L30D_orders AS L30D_orders_treatment,
        t.campaign_period_orders AS campaign_period_orders_treatment,
        t.campaign_period_promo_orders AS campaign_period_promo_orders_treatment,
        t.post_period_orders AS post_period_orders_treatment,
        CAST(NULL AS FLOAT64), CAST(NULL AS FLOAT64),
        CAST(NULL AS FLOAT64), CAST(NULL AS FLOAT64),
        'unmatched' AS source, 'unmatched' AS result_type
    FROM treatment t
    WHERE t.customerid NOT IN (SELECT customerid_treatment FROM exact_matches);
    """

    df_combined = client.query(exact_match_query).to_dataframe()
    if len(df_combined) == 0:
        print(f'  No data for {country} {date_str}')
        return None

    df_exact = df_combined[df_combined['result_type'] == 'exact'].drop(columns=['result_type'])
    df_unmatched = df_combined[df_combined['result_type'] == 'unmatched'].drop(columns=['result_type'])
    del df_combined; gc.collect()

    n_exact = len(df_exact)
    n_unmatched = len(df_unmatched)
    print(f'  Exact match: {n_exact} | Unmatched: {n_unmatched}')

    # ---- STEP 3: KNN fallback (standalone BQ query + local sklearn) ----
    nn_result = pd.DataFrame()
    if n_unmatched > 0:
        # Standalone query for lookalike features — no temp tables, no semicolons
        knn_la_query = f"""
        WITH features AS ({feature_subquery})
        SELECT customerid, optin, L14D_orders, L30D_orders, L90D_orders,
               L180D_orders, L365D_orders, L730D_orders,
               campaign_period_orders, campaign_period_promo_orders,
               post_period_orders
        FROM features
        WHERE treatment = 0 AND {outlier_filter}
        """
        lookalike_knn = client.query(knn_la_query).to_dataframe()

        # Standalone query for unmatched treatment features
        unmatched_ids = df_unmatched['customerid_treatment'].tolist()
        unmatched_ids_str = ', '.join([f"'{cid}'" for cid in unmatched_ids])
        knn_un_query = f"""
        WITH features AS ({feature_subquery})
        SELECT customerid, optin, L14D_orders, L30D_orders, L90D_orders,
               L180D_orders, L365D_orders, L730D_orders,
               campaign_period_orders, campaign_period_promo_orders,
               post_period_orders
        FROM features
        WHERE treatment = 1 AND customerid IN ({unmatched_ids_str})
          AND {outlier_filter}
        """
        unmatched_knn = client.query(knn_un_query).to_dataframe()

        if len(lookalike_knn) > 0 and len(unmatched_knn) > 0:
            scale_cols = ['optin', 'L14D_orders', 'L30D_orders', 'L90D_orders',
                          'L180D_orders', 'L365D_orders', 'L730D_orders']
            # Fill NaNs (e.g. from SAFE_DIVIDE producing NULL) before scaling
            lookalike_knn[scale_cols] = lookalike_knn[scale_cols].fillna(0)
            unmatched_knn[scale_cols] = unmatched_knn[scale_cols].fillna(0)

            scaler = MinMaxScaler()
            la_scaled = scaler.fit_transform(lookalike_knn[scale_cols])
            un_scaled = scaler.transform(unmatched_knn[scale_cols])

            knn = NearestNeighbors(n_neighbors=1)
            knn.fit(la_scaled)
            distances, indices = knn.kneighbors(un_scaled)

            la_matched = lookalike_knn.iloc[indices.flatten()]
            nn_result = pd.DataFrame({
                'customerid_treatment': unmatched_knn['customerid'].values,
                'country': country,
                'optin': unmatched_knn['optin'].values,
                'campaign_start_date': pd.to_datetime(date_str).date(),
                'L30D_orders_treatment': unmatched_knn['L30D_orders'].values,
                'campaign_period_orders_treatment': unmatched_knn['campaign_period_orders'].values,
                'campaign_period_promo_orders_treatment': unmatched_knn['campaign_period_promo_orders'].values,
                'post_period_orders_treatment': unmatched_knn['post_period_orders'].values,
                'L30D_orders_lookalike': la_matched['L30D_orders'].values,
                'campaign_period_orders_lookalike': la_matched['campaign_period_orders'].values,
                'campaign_period_promo_orders_lookalike': la_matched['campaign_period_promo_orders'].values,
                'post_period_orders_lookalike': la_matched['post_period_orders'].values,
                'source': 'nearest_neighbor',
            })
            print(f'  KNN matched: {len(nn_result)} customers')

        del lookalike_knn, unmatched_knn; gc.collect()

    # ---- STEP 4: Combine ----
    parts = [df for df in [df_exact, nn_result] if len(df) > 0]
    if not parts:
        print('  WARNING: No matches produced')
        return None

    final = pd.concat(parts, ignore_index=True)
    final['campaign_name'] = campaign_name
    print(f'  Total matched: {len(final)}')
    del df_exact, nn_result; gc.collect()
    return final

print('V3 matching function loaded (BigQuery-native).')

---
## 7. V2 Customer Lookalike Matching (Stub)

V2 adds `city_type` stratification and GMV metrics on top of V3.

**To enable:**
1. Identify `city_type` source (`fact_segmentation_scv_key.city` → mapping?)
2. Add `city_type` to audience creation query
3. Implement `run_matching_v2()` below
4. Set `RUN_V2 = True` in configuration

In [ ]:
def run_matching_v2(client, country, date_str, city_type, campaign_name,
                    audience_table, look_alike_conditions):
    """V2 Customer Lookalike matching — placeholder.

    TODO: Implement when city_type sourcing is confirmed.
    Should mirror run_matching_v3 but:
      - Filter by city_type in the feature query
      - Include GMV metrics (L30D_GMV, campaign_period_food_total,
        campaign_period_discount)
      - Use LOOK_ALIKE_CONDITIONS_V2
    """
    raise NotImplementedError(
        'V2 matching requires city_type sourcing. '
        'See Section 7 header for implementation steps.'
    )

print('V2 stub loaded. Set RUN_V2 = True after implementing.')

---
## 8. City Lookalike (MAVERICK KNN + CausalImpact)

City-level synthetic control validation. Uses:
- **MAVERICK KNN** to find similar control cities
- **Correlation filter** (≥ 0.8 on weekly order trends)
- **CausalImpact** with **individual** control city time series (not aggregated)

In [ ]:
def get_city_features(client, country, reference_date):
    """City-level features for KNN matching (MAVERICK approach)."""
    query = f"""
    WITH active AS (
        SELECT dr.city,
            SUM(fo.nroforders) AS total_ov,
            COUNT(DISTINCT fo.customerid) AS unique_customers
        FROM `just-data-warehouse.dwh.fact_order` AS fo
        INNER JOIN `just-data-warehouse.dwh.dim_restaurant` AS dr
            ON fo.restaurantid = dr.restaurantid
        WHERE dr.country = '{country}'
          AND DATE(fo.orderdatetime)
              >= DATE_SUB(DATE('{reference_date}'), INTERVAL 90 DAY)
          AND DATE(fo.orderdatetime) < DATE('{reference_date}')
        GROUP BY 1
    ),
    population AS (
        SELECT dz.city,
            SUM(pop_addressable) AS pop_15plus
        FROM `just-data-warehouse.cma_mbr_population.mbr_population` AS pop
        INNER JOIN `just-data-warehouse.dwh.dim_zipcode` AS dz
            ON pop.tableau_map_zipcode = dz.zipcode
        WHERE pop.country = '{country}'
          AND dz.country = '{country}'
          AND year = (
              SELECT MAX(year)
              FROM `just-data-warehouse.cma_mbr_population.mbr_population`
          )
        GROUP BY 1
    ),
    supply AS (
        SELECT dr.city,
            COUNT(dr.restaurantid) AS online_partners,
            SUM(CASE WHEN dr.chain = '(Not available)'
                THEN 1 ELSE 0 END) AS smb_partners,
            SUM(CASE WHEN dr.chain != '(Not available)'
                THEN 1 ELSE 0 END) AS chain_partners
        FROM `just-data-warehouse.dwh.dim_restaurant` AS dr
        WHERE dr.country = '{country}'
          AND dr.onlinestatus
              IN ('Online (temporarily closed)', 'Online (with Menu)')
        GROUP BY 1
    )
    SELECT
        a.city,
        COALESCE(p.pop_15plus, 0) AS pop_15plus,
        a.total_ov,
        a.unique_customers,
        COALESCE(s.online_partners, 0) AS online_partners,
        COALESCE(s.smb_partners, 0) AS smb_partners,
        COALESCE(s.chain_partners, 0) AS chain_partners,
        SAFE_DIVIDE(a.total_ov, p.pop_15plus) AS ov_per_capita
    FROM active AS a
    LEFT JOIN population AS p ON a.city = p.city
    LEFT JOIN supply AS s ON a.city = s.city
    WHERE a.total_ov > 0
    """
    return client.query(query).to_dataframe()


def find_control_cities(df_cities, treatment_city, n_neighbors=25):
    """KNN on city features to find similar control cities (MAVERICK approach)."""
    if treatment_city not in df_cities['city'].values:
        print(f'  WARNING: {treatment_city} not in city features')
        return []

    numerical_cols = df_cities.select_dtypes(include=['number']).columns.tolist()
    scaler = StandardScaler()
    scaled = scaler.fit_transform(df_cities[numerical_cols])

    target_idx = df_cities[df_cities['city'] == treatment_city].index[0]
    target_pos = list(df_cities.index).index(target_idx)

    n = min(n_neighbors + 1, len(df_cities))
    knn = NearestNeighbors(n_neighbors=n, algorithm='auto')
    knn.fit(scaled)
    distances, indices = knn.kneighbors(scaled[target_pos].reshape(1, -1))

    neighbors = []
    for idx in indices.flatten():
        city = df_cities.iloc[idx]['city']
        if city != treatment_city:
            neighbors.append(city)
    return neighbors[:n_neighbors]


def get_daily_orders(client, country, cities, start_date, end_date,
                     segment=None):
    """Daily order time series per city, optionally filtered by segment."""
    city_list = ', '.join([f"'{c}'" for c in cities])

    segment_join = ''
    segment_filter = ''
    if segment:
        segment_join = """
        LEFT JOIN `just-data-warehouse.core_dwh.dim_unique_customer_history` AS ch
            ON fo.customerid = ch.customer_id
            AND DATE_SUB(ch.snapshot_date, INTERVAL 1 DAY) = DATE(fo.orderdatetime)
        LEFT JOIN `just-data-warehouse.dwh.fact_segmentation_scv_key` AS s
            ON ch.scv_key = s.scv_key
            AND DATE_SUB(DATE(fo.orderdatetime), INTERVAL 1 DAY) = s.snapshot_date
        """
        segment_filter = (
            f"AND COALESCE(s.base_segment, 'unknown') = '{segment}'"
        )

    query = f"""
    SELECT DATE(fo.orderdatetime) AS orderdate,
           dr.city,
           SUM(fo.nroforders) AS totalorders
    FROM `just-data-warehouse.dwh.fact_order` AS fo
    INNER JOIN `just-data-warehouse.dwh.dim_restaurant` AS dr
        ON fo.restaurantid = dr.restaurantid
    {segment_join}
    WHERE dr.country = '{country}'
      AND dr.city IN ({city_list})
      AND DATE(fo.orderdatetime) >= DATE('{start_date}')
      AND DATE(fo.orderdatetime) <= DATE('{end_date}')
      {segment_filter}
    GROUP BY 1, 2
    ORDER BY 1, 2
    """
    return client.query(query).to_dataframe()


def apply_correlation_filter(df_orders, treatment_city, control_cities,
                             threshold=0.8):
    """Filter controls by weekly order correlation with treatment city."""
    all_cities = [treatment_city] + control_cities
    df = df_orders[df_orders['city'].isin(all_cities)].copy()

    pivot = df.pivot_table(
        index='orderdate', columns='city', values='totalorders',
    ).fillna(0)
    pivot.index = pd.to_datetime(pivot.index)
    weekly = pivot.resample('W-MON').sum()

    if treatment_city not in weekly.columns:
        print(f'  WARNING: {treatment_city} has no order data')
        return []

    corr = weekly.corr()
    selected = corr[treatment_city]
    passed = selected[
        (selected >= threshold) & (selected.index != treatment_city)
    ]
    print(f'  Correlation filter: {len(passed)}/{len(control_cities)} '
          f'controls pass (>= {threshold})')
    return passed.index.tolist()


def run_city_causal_impact(df_orders, treatment_city, control_cities,
                           pre_period, post_period):
    """
    CausalImpact with INDIVIDUAL control city time series as covariates.
    Key improvement: production aggregates all controls into one bucket.
    This version lets BSTS select and weight the best controls.
    """
    if not CAUSAL_IMPACT_AVAILABLE:
        return None

    all_cities = [treatment_city] + control_cities
    df = df_orders[df_orders['city'].isin(all_cities)].copy()

    pivot = df.pivot_table(
        index='orderdate', columns='city', values='totalorders',
    ).fillna(0)
    pivot.index = pd.to_datetime(pivot.index)

    # Treatment column must be first for CausalImpact
    cols = [treatment_city] + [c for c in pivot.columns if c != treatment_city]
    pivot = pivot[cols]

    pre = [pd.to_datetime(d) for d in pre_period]
    post = [pd.to_datetime(d) for d in post_period]

    try:
        ci = CausalImpact(
            pivot, pre, post, model_args={'fit_method': 'hmc'},
        )
        summary = ci.summary_data
        actual = summary.loc['actual', 'cumulative']
        predicted = summary.loc['predicted', 'cumulative']
        uplift = (actual / predicted - 1) if predicted > 0 else np.nan
        return {
            'actual_orders': actual,
            'predicted_orders': predicted,
            'uplift': uplift,
        }
    except Exception as e:
        print(f'  CausalImpact error: {e}')
        return None

print('City Lookalike functions loaded.')

---
## 9. Orchestrators

In [ ]:
def run_customer_aa_tests(client, countries, time_windows, n_seeds,
                          treatment_share, audience_table,
                          variant, look_alike_conditions):
    """Orchestrate customer-level A/A tests — memory-optimized.
    Frees memory between runs with gc.collect().
    """
    all_results = []
    total = sum(len(time_windows.get(c, [])) for c in countries) * n_seeds
    run_count = 0

    for country in countries:
        for window in time_windows.get(country, []):
            for seed in range(n_seeds):
                run_count += 1
                campaign_name = (
                    f"{AA_CAMPAIGN_PREFIX}_{country}_"
                    f"{window['start']}_seed{seed}"
                )
                print(f'\n=== [{variant}] Run {run_count}/{total}: '
                      f'{campaign_name} ===')

                try:
                    df_aud, cname = create_customer_aa_audience(
                        client, country, window, seed, treatment_share,
                    )
                    upload_aa_audience(client, df_aud, audience_table)

                    if variant == 'V3':
                        result = run_matching_v3(
                            client, country, window['start'], cname,
                            audience_table, look_alike_conditions,
                        )
                    else:
                        result = None
                        print('  V2 matching not yet implemented')

                    if result is not None and len(result) > 0:
                        # Merge segments from audience DataFrame
                        seg_map = df_aud[['customerid', 'base_segment']].drop_duplicates()
                        result = result.merge(
                            seg_map, left_on='customerid_treatment',
                            right_on='customerid', how='left',
                        ).drop(columns=['customerid'], errors='ignore')
                        result['base_segment'] = result['base_segment'].fillna('unknown')

                        result['seed'] = seed
                        result['window_start'] = window['start']
                        result['window_end'] = window['end']
                        result['variant'] = variant
                        all_results.append(result)
                    else:
                        print(f'  WARNING: No results for {cname}')

                except Exception as e:
                    print(f'  ERROR: {e}')
                    import traceback
                    traceback.print_exc()

                finally:
                    try:
                        cleanup_aa_audience(
                            client, audience_table, campaign_name,
                        )
                    except Exception:
                        pass
                    # Free memory between runs
                    del df_aud
                    gc.collect()

    if all_results:
        return pd.concat(all_results, ignore_index=True)
    print('ERROR: No customer-level results collected.')
    return pd.DataFrame()

print('Customer orchestrator loaded (memory-optimized).')

In [ ]:
def run_city_aa_tests(client, countries, time_windows,
                      city_candidates, n_neighbors, corr_threshold):
    """Orchestrate city-level A/A tests (MAVERICK + CausalImpact)."""
    if not CAUSAL_IMPACT_AVAILABLE:
        print('CausalImpact not installed — skipping City Lookalike A/A.')
        return pd.DataFrame()

    all_results = []

    for country in countries:
        candidates = city_candidates.get(country, [])
        if not candidates:
            print(f'No City Lookalike candidates for {country} — skipping')
            continue

        for window in time_windows.get(country, []):
            pre_start = (
                pd.to_datetime(window['start'])
                - pd.Timedelta(days=90)
            ).strftime('%Y-%m-%d')
            pre_end = (
                pd.to_datetime(window['start'])
                - pd.Timedelta(days=1)
            ).strftime('%Y-%m-%d')

            print(f'\n=== [CITY] Getting city features for {country} '
                  f'(ref: {window["start"]}) ===')
            df_cities = get_city_features(
                client, country, window['start'],
            )
            print(f'  {len(df_cities)} cities with features')

            for treatment_city in candidates:
                print(f'\n=== [CITY] {treatment_city} | '
                      f'{window["start"]} - {window["end"]} ===')

                controls = find_control_cities(
                    df_cities, treatment_city, n_neighbors,
                )
                if not controls:
                    continue
                print(f'  KNN found {len(controls)} candidate controls')

                all_cities = [treatment_city] + controls
                df_orders = get_daily_orders(
                    client, country, all_cities,
                    pre_start, window['post_end'],
                )

                df_pre = df_orders[df_orders['orderdate'] <= pre_end]
                filtered = apply_correlation_filter(
                    df_pre, treatment_city, controls, corr_threshold,
                )
                if not filtered:
                    continue

                pre_period = [pre_start, pre_end]
                post_period = [window['start'], window['end']]

                # Total level
                result = run_city_causal_impact(
                    df_orders, treatment_city, filtered,
                    pre_period, post_period,
                )
                if result:
                    all_results.append({
                        'country': country,
                        'treatment_city': treatment_city,
                        'window_start': window['start'],
                        'window_end': window['end'],
                        'n_controls': len(filtered),
                        'base_segment': 'total',
                        'variant': 'CITY',
                        **result,
                    })

                # Per segment
                for segment in BASE_SEGMENTS:
                    if segment == 'unknown':
                        continue
                    df_seg = get_daily_orders(
                        client, country, all_cities,
                        pre_start, window['post_end'],
                        segment=segment,
                    )
                    if df_seg.empty:
                        continue

                    seg_filtered = apply_correlation_filter(
                        df_seg[df_seg['orderdate'] <= pre_end],
                        treatment_city, controls, corr_threshold,
                    )
                    if not seg_filtered:
                        continue

                    seg_result = run_city_causal_impact(
                        df_seg, treatment_city, seg_filtered,
                        pre_period, post_period,
                    )
                    if seg_result:
                        all_results.append({
                            'country': country,
                            'treatment_city': treatment_city,
                            'window_start': window['start'],
                            'window_end': window['end'],
                            'n_controls': len(seg_filtered),
                            'base_segment': segment,
                            'variant': 'CITY',
                            **seg_result,
                        })

    if all_results:
        return pd.DataFrame(all_results)
    print('No city-level results collected.')
    return pd.DataFrame()

print('City orchestrator loaded.')

---
## 10. Bias Metrics & Statistical Validation

In [ ]:
def _add_uplift_columns(agg):
    """Add uplift, incremental, and coverage columns."""
    agg['pre_period_uplift'] = safe_divide(
        agg['L30D_treatment'], agg['L30D_lookalike'],
    ) - 1
    agg['campaign_period_uplift'] = safe_divide(
        agg['campaign_treatment'], agg['campaign_lookalike'],
    ) - 1
    agg['post_period_uplift'] = safe_divide(
        agg['post_treatment'], agg['post_lookalike'],
    ) - 1
    agg['campaign_incr_orders'] = (
        agg['campaign_treatment'] - agg['campaign_lookalike']
    )
    agg['post_incr_orders'] = (
        agg['post_treatment'] - agg['post_lookalike']
    )
    agg['exact_match_pct'] = safe_divide(
        agg['n_exact'], agg['n_customers'],
    )


def calculate_customer_bias_metrics(df):
    """Calculate uplift metrics for customer-level results."""
    if 'base_segment' not in df.columns:
        df['base_segment'] = 'unknown'

    agg_spec = {
        'n_customers': ('customerid_treatment', 'nunique'),
        'n_exact': ('source', lambda x: (x == 'exact_match').sum()),
        'n_knn': ('source', lambda x: (x == 'nearest_neighbor').sum()),
        'L30D_treatment': ('L30D_orders_treatment', 'sum'),
        'L30D_lookalike': ('L30D_orders_lookalike', 'sum'),
        'campaign_treatment': ('campaign_period_orders_treatment', 'sum'),
        'campaign_lookalike': ('campaign_period_orders_lookalike', 'sum'),
        'campaign_promo_treatment': ('campaign_period_promo_orders_treatment', 'sum'),
        'campaign_promo_lookalike': ('campaign_period_promo_orders_lookalike', 'sum'),
        'post_treatment': ('post_period_orders_treatment', 'sum'),
        'post_lookalike': ('post_period_orders_lookalike', 'sum'),
    }

    # Total level
    total_group = ['window_start', 'seed', 'country', 'variant']
    total_agg = df.groupby(total_group).agg(**agg_spec).reset_index()
    total_agg['base_segment'] = 'total'
    _add_uplift_columns(total_agg)

    # Segment level
    seg_group = total_group + ['base_segment']
    seg_agg = df.groupby(seg_group).agg(**agg_spec).reset_index()
    _add_uplift_columns(seg_agg)

    return pd.concat([total_agg, seg_agg], ignore_index=True)


def run_statistical_validation(metrics_df, variant='V3', segment='total'):
    """Run t-test + bootstrap CI for each uplift period."""
    df_sub = metrics_df[
        (metrics_df['variant'] == variant)
        & (metrics_df['base_segment'] == segment)
    ]
    results = {}
    for period in ['pre_period_uplift', 'campaign_period_uplift',
                    'post_period_uplift']:
        values = df_sub[period].dropna().values
        sv = statistical_validation_single(values, label=period)
        sv['verdict'] = assess_verdict(
            abs(sv['mean']) if not np.isnan(sv['mean']) else np.nan,
            sv['ci_contains_zero'],
        )
        results[period] = sv
    return pd.DataFrame(results).T


def generate_scorecard(metrics_df, variants=None):
    """Generate pass/fail scorecard across all variants and segments."""
    if variants is None:
        variants = metrics_df['variant'].unique()

    all_rows = []
    for variant in variants:
        segments = metrics_df[
            metrics_df['variant'] == variant
        ]['base_segment'].unique()
        for segment in segments:
            st = run_statistical_validation(metrics_df, variant, segment)
            for period, row in st.iterrows():
                all_rows.append({
                    'variant': variant, 'segment': segment,
                    'period': period, 'mean_uplift': row['mean'],
                    'p_value': row['p_value'],
                    'ci_lower': row['ci_lower'],
                    'ci_upper': row['ci_upper'],
                    'ci_contains_zero': row['ci_contains_zero'],
                    'verdict': row['verdict'], 'n_runs': row['n'],
                })

    df_scorecard = pd.DataFrame(all_rows)
    verdicts = df_scorecard['verdict'].values
    if 'HARD_FAIL' in verdicts:
        overall = 'HARD_FAIL'
    elif 'WARNING' in verdicts:
        overall = 'WARNING'
    elif 'INSUFFICIENT_DATA' in verdicts:
        overall = 'INSUFFICIENT_DATA'
    else:
        overall = 'PASS'

    return df_scorecard, overall


def stability_analysis(metrics_df, variant='V3'):
    """Test bias consistency across time windows (Levene + Kruskal-Wallis)."""
    df = metrics_df[
        (metrics_df['variant'] == variant)
        & (metrics_df['base_segment'] == 'total')
    ]
    window_stats = df.groupby('window_start').agg({
        'campaign_period_uplift': ['mean', 'std', 'count'],
        'exact_match_pct': ['mean'],
    })
    print(f'\nCross-window statistics ({variant}):')
    print(window_stats.to_string())

    groups = [
        g['campaign_period_uplift'].dropna().values
        for _, g in df.groupby('window_start')
    ]
    groups = [g for g in groups if len(g) >= 2]

    if len(groups) >= 2:
        lev_stat, lev_p = stats.levene(*groups)
        print(f'\nLevene (equal variance): stat={lev_stat:.4f}, p={lev_p:.4f}')
        print('  ' + ('WARNING: Variance differs.' if lev_p < 0.05 else 'PASS'))

        kw_stat, kw_p = stats.kruskal(*groups)
        print(f'Kruskal-Wallis (equal medians): stat={kw_stat:.4f}, p={kw_p:.4f}')
        print('  ' + ('WARNING: Medians differ.' if kw_p < 0.05 else 'PASS'))

    return window_stats

print('Metrics & validation functions loaded.')

---
## 11. Visualisations

In [ ]:
def plot_uplift_distributions(metrics_df, variant='V3'):
    df = metrics_df[
        (metrics_df['variant'] == variant) & (metrics_df['base_segment'] == 'total')
    ]
    periods = [
        ('pre_period_uplift', 'Pre-Period'),
        ('campaign_period_uplift', 'Campaign-Period'),
        ('post_period_uplift', 'Post-Period'),
    ]
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for i, (col, title) in enumerate(periods):
        ax = axes[i]
        values = df[col].dropna()
        if len(values) == 0:
            ax.set_title(f'{title} (no data)')
            continue
        sns.histplot(values, kde=True, ax=ax, bins=20, color='steelblue')
        ax.axvline(x=0, color='red', ls='--', lw=2, label='Expected (0)')
        ax.axvline(x=values.mean(), color='orange', ls='-', lw=2,
                   label=f'Mean: {values.mean():.4f}')
        ax.set_title(title)
        ax.set_xlabel('Uplift')
        ax.legend(fontsize=9)
    plt.suptitle(f'A/A Test: {variant} Uplift Distribution', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()


def plot_uplift_by_window(metrics_df, variant='V3'):
    df = metrics_df[
        (metrics_df['variant'] == variant) & (metrics_df['base_segment'] == 'total')
    ]
    periods = [
        ('pre_period_uplift', 'Pre-Period'),
        ('campaign_period_uplift', 'Campaign-Period'),
        ('post_period_uplift', 'Post-Period'),
    ]
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for i, (col, title) in enumerate(periods):
        ax = axes[i]
        df.boxplot(column=col, by='window_start', ax=ax)
        ax.axhline(y=0, color='red', ls='--', lw=1.5)
        ax.set_title(title)
        ax.set_xlabel('Window Start')
        ax.set_ylabel('Uplift')
        plt.sca(ax)
        plt.xticks(rotation=45)
    plt.suptitle(f'A/A Test: {variant} Stability Across Windows', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()


def plot_segment_heatmap(metrics_df, variant='V3'):
    df = metrics_df[
        (metrics_df['variant'] == variant) & (metrics_df['base_segment'] != 'total')
    ]
    if len(df) == 0:
        print('No segment-level data to plot.')
        return
    periods = ['pre_period_uplift', 'campaign_period_uplift', 'post_period_uplift']
    heatmap_data = df.groupby('base_segment')[periods].mean()
    fig, ax = plt.subplots(figsize=(10, 7))
    sns.heatmap(
        heatmap_data, annot=True, fmt='.4f', cmap='RdYlGn_r',
        center=0, ax=ax, linewidths=0.5,
        xticklabels=['Pre-Period', 'Campaign', 'Post-Period'],
    )
    ax.set_title(f'{variant}: Mean Uplift by Segment (0 = no bias)')
    ax.set_ylabel('Base Segment')
    plt.tight_layout()
    plt.show()


def plot_city_results(df_city_results):
    if df_city_results.empty:
        print('No city-level results to plot.')
        return
    df_total = df_city_results[df_city_results['base_segment'] == 'total']
    fig, ax = plt.subplots(figsize=(12, 6))
    labels = [
        f"{r['treatment_city']}\n{r['window_start']}"
        for _, r in df_total.iterrows()
    ]
    uplifts = df_total['uplift'].values
    colors = [
        'green' if abs(u) <= BIAS_THRESHOLD_WARN
        else 'orange' if abs(u) <= BIAS_THRESHOLD_HARD
        else 'red'
        for u in uplifts
    ]
    ax.bar(range(len(labels)), uplifts, color=colors)
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha='right')
    ax.axhline(y=0, color='black', ls='-', lw=0.5)
    ax.axhline(y=BIAS_THRESHOLD_WARN, color='orange', ls=':', alpha=0.7)
    ax.axhline(y=-BIAS_THRESHOLD_WARN, color='orange', ls=':', alpha=0.7)
    ax.axhline(y=BIAS_THRESHOLD_HARD, color='red', ls=':', alpha=0.7)
    ax.axhline(y=-BIAS_THRESHOLD_HARD, color='red', ls=':', alpha=0.7)
    ax.set_ylabel('Uplift')
    ax.set_title('City Lookalike A/A: Uplift by Treatment City & Window')
    plt.tight_layout()
    plt.show()

print('Visualization functions loaded.')

---
## 12. Run Everything

In [ ]:
print('=' * 60)
print('STEP 1: Campaign Overlap Validation')
print('=' * 60)
df_overlap = validate_no_campaign_overlap(client, COUNTRIES, TIME_WINDOWS)

In [ ]:
df_customer_results = pd.DataFrame()
if RUN_V3:
    print('\n' + '=' * 60)
    print('STEP 2: Running V3 Customer Lookalike A/A')
    print('=' * 60)
    start_time = time.time()

    df_v3 = run_customer_aa_tests(
        client=client, countries=COUNTRIES, time_windows=TIME_WINDOWS,
        n_seeds=N_SEEDS, treatment_share=TREATMENT_SHARE,
        audience_table=AA_AUDIENCE_TABLE,
        variant='V3', look_alike_conditions=LOOK_ALIKE_CONDITIONS_V3,
    )

    elapsed = (time.time() - start_time) / 60
    print(f'\nV3 complete in {elapsed:.1f} minutes. Rows: {len(df_v3)}')
    df_customer_results = pd.concat(
        [df_customer_results, df_v3], ignore_index=True,
    )

In [ ]:
df_city_results = pd.DataFrame()
if RUN_CITY:
    print('\n' + '=' * 60)
    print('STEP 3: Running City Lookalike A/A')
    print('=' * 60)
    start_time = time.time()

    df_city_results = run_city_aa_tests(
        client=client, countries=COUNTRIES, time_windows=TIME_WINDOWS,
        city_candidates=CITY_TREATMENT_CANDIDATES,
        n_neighbors=CITY_KNN_NEIGHBORS,
        corr_threshold=CITY_CORRELATION_THRESHOLD,
    )

    elapsed = (time.time() - start_time) / 60
    print(f'\nCity Lookalike complete in {elapsed:.1f} minutes. '
          f'Rows: {len(df_city_results)}')

---
## 13. Analysis & Reporting

In [ ]:
df_metrics = pd.DataFrame()
if len(df_customer_results) > 0:
    print('=' * 60)
    print('STEP 4: Calculating Bias Metrics')
    print('=' * 60)
    df_metrics = calculate_customer_bias_metrics(df_customer_results)
    print(f'Metrics calculated for {len(df_metrics)} rows')

    df_total = df_metrics[df_metrics['base_segment'] == 'total']
    for variant in df_total['variant'].unique():
        df_v = df_total[df_total['variant'] == variant]
        print(f'\n{variant} total-level summary:')
        for col in ['pre_period_uplift', 'campaign_period_uplift', 'post_period_uplift']:
            vals = df_v[col].dropna()
            if len(vals) > 0:
                print(f'  {col}: mean={vals.mean():+.4f} std={vals.std():.4f}')

In [ ]:
df_scorecard = pd.DataFrame()
overall_verdict = 'NO_DATA'
if len(df_metrics) > 0:
    print('=' * 60)
    print('STEP 5: Statistical Validation & Pass/Fail')
    print('=' * 60)
    df_scorecard, overall_verdict = generate_scorecard(df_metrics)
    print('\nScorecard (total-level):')
    total_card = df_scorecard[df_scorecard['segment'] == 'total']
    print(total_card[
        ['variant', 'period', 'mean_uplift', 'p_value',
         'ci_lower', 'ci_upper', 'verdict', 'n_runs']
    ].to_string(index=False))

In [ ]:
if len(df_metrics) > 0:
    print('=' * 60)
    print('STEP 6: Stability Analysis')
    print('=' * 60)
    for variant in df_metrics['variant'].unique():
        stability_analysis(df_metrics, variant)

In [ ]:
if len(df_metrics) > 0:
    for variant in df_metrics['variant'].unique():
        plot_uplift_distributions(df_metrics, variant)
        plot_uplift_by_window(df_metrics, variant)
        plot_segment_heatmap(df_metrics, variant)

if len(df_city_results) > 0:
    plot_city_results(df_city_results)

In [ ]:
if len(df_city_results) > 0:
    print('=' * 60)
    print('City Lookalike Results')
    print('=' * 60)
    for _, row in df_city_results.iterrows():
        uplift = row['uplift']
        verdict = assess_verdict(
            abs(uplift) if not np.isnan(uplift) else np.nan, None,
        )
        print(f"  {row['treatment_city']:15s} | {row['window_start']} | "
              f"segment={row['base_segment']:12s} | "
              f"uplift={uplift:+.4f} | {verdict}")

---
## 14. Summary Report

In [ ]:
print('=' * 60)
print('A/A TEST SUMMARY REPORT')
print('=' * 60)

print(f'\nConfiguration:')
print(f'  Markets:          {COUNTRIES}')
print(f'  Time windows:     {sum(len(TIME_WINDOWS.get(c, [])) for c in COUNTRIES)}')
print(f'  Seeds per window: {N_SEEDS}')
print(f'  Techniques:       V3={RUN_V3}, V2={RUN_V2}, City={RUN_CITY}')

print(f'\nThresholds:')
print(f'  Hard fail: |uplift| > {BIAS_THRESHOLD_HARD:.0%}')
print(f'  Warning:   |uplift| > {BIAS_THRESHOLD_WARN:.0%}')

if len(df_metrics) > 0:
    df_total = df_metrics[df_metrics['base_segment'] == 'total']
    print(f'\nCustomer-Level Results ({len(df_total)} runs):')
    for variant in df_total['variant'].unique():
        df_v = df_total[df_total['variant'] == variant]
        print(f'\n  {variant}:')
        for col, label in [
            ('pre_period_uplift', 'Pre-period'),
            ('campaign_period_uplift', 'Campaign'),
            ('post_period_uplift', 'Post-period'),
        ]:
            vals = df_v[col].dropna()
            if len(vals) > 0:
                print(f'    {label}: mean={vals.mean():+.4f} '
                      f'std={vals.std():.4f} '
                      f'range=[{vals.min():.4f}, {vals.max():.4f}]')
        cov = df_v['exact_match_pct'].dropna()
        if len(cov) > 0:
            print(f'    Exact match coverage: {cov.mean():.1%} '
                  f'(std: {cov.std():.1%})')

if len(df_city_results) > 0:
    df_city_total = df_city_results[df_city_results['base_segment'] == 'total']
    print(f'\nCity-Level Results ({len(df_city_total)} runs):')
    uplifts = df_city_total['uplift'].dropna()
    if len(uplifts) > 0:
        print(f'  Mean uplift: {uplifts.mean():+.4f}')
        print(f'  Std:         {uplifts.std():.4f}')
        print(f'  Range:       [{uplifts.min():.4f}, {uplifts.max():.4f}]')

print(f'\n{"=" * 60}')
print(f'  OVERALL VERDICT: {overall_verdict}')
print(f'{"=" * 60}')

if overall_verdict == 'PASS':
    print('\nSynthetic control methodologies show no statistically '
          'significant bias.')
    print('Proceed to Phase 2 optimisation with confidence.')
elif overall_verdict == 'WARNING':
    print('\nBorderline bias detected. Review per-segment results.')
    print('Consider excluding high-bias segments or adjusting '
          'matching conditions.')
elif overall_verdict == 'HARD_FAIL':
    print('\nSignificant inherent bias detected. Do NOT use these '
          'techniques for production evaluation without further '
          'optimisation.')